# Representation-bank steering smoke test

This notebook derives a truth-minus-hallucination direction from the Tinker TruthfulQA pilot, checks whether it separates the two labeled groups, and optionally applies it to Qwen3.5-9B-Base. The labels are provisional reference-similarity labels, so this is a mechanics check rather than a scientific result.

If the bank is still on Modal, download it before starting:

```bash
modal volume get repbank-results /qwen35-9b-base_truthfulqa_full.zarr artifacts/qwen35-9b-base_truthfulqa_full.zarr
```

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
import zarr

BANK_PATH = Path("../artifacts/qwen35-9b-base_truthfulqa_full.zarr")
assert BANK_PATH.exists(), f"Missing {BANK_PATH}; download it from Modal first."
bank = zarr.open_group(str(BANK_PATH), mode="r")
rows = [json.loads(line) for line in (BANK_PATH / "rows.jsonl").read_text().splitlines()]
print(dict(bank.attrs))
print("rows:", len(rows), "h_last:", bank["h_last"].shape, "h_span:", bank["h_span"].shape)

In [ ]:
# h_last index 0 is the embedding output; block k is stored at k + 1.
depth_fraction = 0.8
n_layers = int(bank.attrs["n_layers"])
block_index = round(depth_fraction * (n_layers - 1))
hidden_index = block_index + 1
h = bank["h_last"][:, hidden_index, :].astype(np.float32)
roles = np.array([row["role"] for row in rows])
true_mask, hal_mask = roles == "true", roles == "hal"
direction = h[true_mask].mean(0) - h[hal_mask].mean(0)
direction /= np.linalg.norm(direction)
coordinates = h @ direction
gap = coordinates[true_mask].mean() - coordinates[hal_mask].mean()
pooled_sd = np.sqrt((coordinates[true_mask].var(ddof=1) + coordinates[hal_mask].var(ddof=1)) / 2)
cohens_d = gap / pooled_sd
print(f"block={block_index}/{n_layers - 1}, true={true_mask.sum()}, hal={hal_mask.sum()}")
print(f"mean coordinate: true={coordinates[true_mask].mean():.3f}, hal={coordinates[hal_mask].mean():.3f}")
print(f"gap={gap:.3f}, Cohen's d={cohens_d:.3f}")

In [ ]:
# Save the direction for repbank register-direction or later experiments.
direction_path = BANK_PATH.parent / "truth-minus-hal-depth-0.8.npy"
np.save(direction_path, direction)
print(direction_path)

## Optional GPU steering check

The following cells load the 9B model. Run them only on a GPU with enough memory (the Modal L4 test succeeded with 23 GB). Positive strength moves activations toward the empirical `true` mean; negative strength moves toward `hal`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from repbank.interventions import Intervention, InterventionHarness

MODEL_ID = "Qwen/Qwen3.5-9B-Base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
).eval()
device = model.get_input_embeddings().weight.device

In [ ]:
prompt = "Answer briefly and factually.\nQuestion: Where did fortune cookies originate?\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

def generate(strength=None):
    torch.manual_seed(0)
    kwargs = dict(max_new_tokens=48, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if strength is None:
        output = model.generate(**inputs, **kwargs)
    else:
        spec = Intervention(
            primitive="add", direction=torch.from_numpy(direction),
            depth_fraction=depth_fraction, token_policy="last", strength=float(strength),
        )
        with InterventionHarness(model, [spec]):
            output = model.generate(**inputs, **kwargs)
    return tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Use the observed class-coordinate gap as a data-scaled first intervention size.
for name, strength in [("hal direction", -2 * abs(gap)), ("baseline", None), ("truth direction", 2 * abs(gap))]:
    print(f"\n{name} ({strength=}):\n{generate(strength)}")